# DWS SnowCamp — Day 2: Dashboards, Governance & AI

## From Mart Tables to Interactive Data Products

**Duration**: ~1.5 hours  |  **Level**: Intermediate  |  **Account**: Same Snowflake Trial as Day 1

---

Welcome back! Yesterday you built a complete data pipeline — from raw synthetic data through to tested, scheduled dbt transformations. Today we will take those mart tables and turn them into **things people can actually use**: interactive dashboards, governed data products, and monitoring for platform administration.

By the end of today you will have:

| What | Why It Matters |
|------|---------------|
| An interactive Streamlit dashboard | Self-service analytics for portfolio managers and clients |
| Horizon Catalog tags and an Internal Marketplace listing | Data governance and discoverability at scale |
| Resource monitors and query analysis | Cost control and performance optimisation |
| Experience with Cortex Code | AI-assisted development inside Snowflake |

| Day 2 | Topic | Duration |
|-------|-------|----------|
| 1 | Streamlit Dashboard | 20 min |
| 2 | Horizon Catalog & Internal Marketplace | 40 min |
| 3 | Platform Administration & Monitoring | 20 min |
| 4 | Cortex Code: What's Next | 10 min |

> **Important**: This notebook runs in a **Snowflake Notebook** (Projects → Notebooks), which includes a built-in Streamlit runtime. This is what lets us render interactive widgets directly in notebook cells.

> **Reference**: [Snowflake Notebooks](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks)

### Prerequisite Check

Before we begin, let's make sure all the objects from Day 1 are in place. The query below checks that the key tables and views exist. If anything is missing, re-run the Day 1 notebook first.

In [ ]:
-- Verify Day 1 objects exist
SELECT 'F_POSITIONS_DAILY'        AS object_name, COUNT(*) AS row_count FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY
UNION ALL SELECT 'F_PERFORMANCE_DAILY',      COUNT(*) FROM SNOWCAMP_LAB.MARTS.F_PERFORMANCE_DAILY
UNION ALL SELECT 'F_HOLDINGS_WITH_MARKET_DATA', COUNT(*) FROM SNOWCAMP_LAB.MARTS.F_HOLDINGS_WITH_MARKET_DATA
UNION ALL SELECT 'D_PORTFOLIO',              COUNT(*) FROM SNOWCAMP_LAB.MARTS.D_PORTFOLIO
UNION ALL SELECT 'D_CLIENT',                 COUNT(*) FROM SNOWCAMP_LAB.MARTS.D_CLIENT
ORDER BY object_name;

---
## Part 1: Streamlit Dashboard (20 min)

Now let's turn those carefully built mart tables into something a portfolio manager or client would actually see: an **interactive dashboard**. Streamlit in Snowflake lets you build rich web applications entirely in Python, and because this is a Snowflake Notebook, you can render Streamlit widgets **directly in cells**.

The dashboard we build will show:
- **KPI metrics** — total clients, portfolios, and AUM at a glance
- **AUM by region** — how assets are distributed geographically
- **Mark-to-market comparison** — synthetic vs real NASDAQ prices from the Marketplace
- **Holdings detail** — a searchable table of individual positions

This is the kind of dashboard a DWS relationship manager might open each morning to check on their clients' portfolios.

> **Reference**: [Streamlit in Snowflake](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit) | [CREATE STREAMLIT](https://docs.snowflake.com/en/sql-reference/sql/create-streamlit)

### 1a. Inline Streamlit Dashboard

The cell below builds a complete dashboard using Streamlit widgets. Because we are in a Snowflake Notebook with the Streamlit runtime, these widgets render as interactive elements — metrics cards, bar charts, and data tables — right here in the notebook.

Take a moment to read through the code. Notice how we use `session.sql(...)` to query our mart tables, convert the results to Pandas DataFrames, and then pass them to Streamlit components. This is the Snowpark + Streamlit pattern you will use for all your dashboards.

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("DWS Client Reporting Dashboard")
st.caption("Built with Streamlit in Snowflake | SnowCamp Hands-On Lab")

# ---- KPI Metrics ----
kpi = session.sql('''
    SELECT COUNT(DISTINCT f.client_id) AS clients,
           COUNT(DISTINCT f.portfolio_id) AS portfolios,
           ROUND(SUM(f.market_value), 0) AS aum
    FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY f
    WHERE f.as_of_date = (SELECT MAX(as_of_date) FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY)
''').to_pandas()
c1, c2, c3 = st.columns(3)
c1.metric("Clients", f"{kpi['CLIENTS'].iloc[0]:,}")
c2.metric("Portfolios", f"{kpi['PORTFOLIOS'].iloc[0]:,}")
c3.metric("Total AUM", f"${kpi['AUM'].iloc[0]:,.0f}")

# ---- AUM by Region ----
st.subheader("AUM by Region")
aum_region = session.sql('''
    SELECT c.region, ROUND(SUM(f.market_value), 0) AS aum
    FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY f
    JOIN SNOWCAMP_LAB.MARTS.D_CLIENT c ON f.client_id = c.client_id
    WHERE f.as_of_date = (SELECT MAX(as_of_date) FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY)
    GROUP BY c.region ORDER BY aum DESC
''').to_pandas()
st.bar_chart(aum_region.set_index("REGION"))

# ---- Mark-to-Market: Real vs Synthetic ----
st.subheader("Mark-to-Market: Real vs Synthetic (Snowflake Marketplace)")
st.caption("Synthetic holdings enriched with real NASDAQ closing prices via Snowflake Marketplace")
mtm = session.sql('''
    SELECT ticker, security_name,
           ROUND(SUM(synthetic_market_value), 0) AS synthetic_val,
           ROUND(SUM(mark_to_market_value), 0)   AS market_val
    FROM SNOWCAMP_LAB.MARTS.F_HOLDINGS_WITH_MARKET_DATA
    WHERE market_close_price IS NOT NULL
      AND as_of_date = (SELECT MAX(as_of_date) FROM SNOWCAMP_LAB.MARTS.F_HOLDINGS_WITH_MARKET_DATA)
    GROUP BY ticker, security_name
    ORDER BY market_val DESC LIMIT 15
''').to_pandas()
if not mtm.empty:
    st.bar_chart(mtm.set_index("TICKER")[["SYNTHETIC_VAL", "MARKET_VAL"]])

# ---- Top Holdings ----
st.subheader("Top Holdings")
holdings = session.sql('''
    SELECT f.portfolio_name, c.client_name, f.ticker, f.security_name,
           f.sector, f.asset_class, f.quantity,
           ROUND(f.market_value, 2) AS market_value,
           ROUND(f.portfolio_weight * 100, 2) AS weight_pct
    FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY f
    JOIN SNOWCAMP_LAB.MARTS.D_CLIENT c ON f.client_id = c.client_id
    WHERE f.as_of_date = (SELECT MAX(as_of_date) FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY)
    ORDER BY f.market_value DESC LIMIT 50
''').to_pandas()
st.dataframe(holdings, use_container_width=True)

### 1b. Deploy as a Standalone Streamlit App

The inline dashboard above is great for exploration, but for production use you want a **standalone Streamlit app** — a dedicated URL that anyone with the right role can access, with sidebar filters and a polished layout.

The lab's GitHub repo includes a ready-to-deploy app at `streamlit/client_reporting_app.py` with interactive region and asset-class filters, cumulative performance charts, and the mark-to-market comparison.

After deploying, you can find it in **Projects → Streamlit** in Snowsight, or click the URL from `SHOW STREAMLITS`.

> **Reference**: [CREATE STREAMLIT](https://docs.snowflake.com/en/sql-reference/sql/create-streamlit) | [Streamlit in Snowflake Overview](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit)

In [ ]:
-- Make sure we have the latest repo content
ALTER GIT REPOSITORY SNOWCAMP_LAB.GITREPO.SNOWCAMP_GIT_REPO FETCH;

-- List the Streamlit app files
LS @SNOWCAMP_LAB.GITREPO.SNOWCAMP_GIT_REPO/branches/main/streamlit/;

-- Deploy the standalone Streamlit app
CREATE OR REPLACE STREAMLIT SNOWCAMP_LAB.ANALYTICS.CLIENT_REPORTING_APP
    ROOT_LOCATION  = '@SNOWCAMP_LAB.GITREPO.SNOWCAMP_GIT_REPO/branches/main/streamlit'
    MAIN_FILE      = 'client_reporting_app.py'
    QUERY_WAREHOUSE = WH_LAB
    TITLE          = 'DWS Client Reporting'
    COMMENT        = 'SnowCamp lab - interactive client reporting dashboard';

SHOW STREAMLITS IN SCHEMA SNOWCAMP_LAB.ANALYTICS;

---
## Part 2: Horizon Catalog & Internal Marketplace (40 min)

Yesterday we consumed data from the Marketplace. Now let's flip the script and become **data publishers**. In a large organisation like DWS, different teams produce data products that other teams need — risk analytics for compliance, client data for relationship managers, performance data for fund reporting.

Snowflake's **Horizon Catalog** provides two key capabilities:
1. **Object tagging** — attach business metadata (domain, confidentiality level, data steward) to tables so that consumers can discover and understand data products
2. **Internal Marketplace** — publish your data products as shareable listings that other teams can install with a click

The combination of tags + shares means you can implement a full **data mesh** architecture: each team owns their data product, tags it with governance metadata, and publishes it for consumption.

> **Reference**: [Snowflake Horizon](https://docs.snowflake.com/en/user-guide/governance-horizon) | [Object Tagging](https://docs.snowflake.com/en/user-guide/object-tagging)

### 2a. Object Tagging

Tags are key-value pairs that you attach to any Snowflake object — databases, schemas, tables, even individual columns. They power **data classification**, **lineage**, and **policy enforcement**.

We will create three tags:
- **`DATA_DOMAIN`** — Which business area owns this data? (Client Reporting, Risk, etc.)
- **`CONFIDENTIALITY`** — How sensitive is this data? (Public, Internal, Confidential, Restricted)
- **`DATA_STEWARD`** — Who is responsible for data quality?

These tags are visible in the Snowsight UI, searchable via SQL, and can drive **tag-based masking policies** — for example, automatically masking columns tagged as `Confidential` for users without the right role.

In [ ]:
USE SCHEMA SNOWCAMP_LAB.MARTS;

CREATE OR REPLACE TAG DATA_DOMAIN
    ALLOWED_VALUES 'Client Reporting', 'Risk', 'Compliance', 'ESG'
    COMMENT = 'Business domain that owns the data product';

CREATE OR REPLACE TAG CONFIDENTIALITY
    ALLOWED_VALUES 'Public', 'Internal', 'Confidential', 'Restricted'
    COMMENT = 'Data classification level';

CREATE OR REPLACE TAG DATA_STEWARD
    COMMENT = 'Team or person responsible for data quality';

ALTER TABLE F_POSITIONS_DAILY SET TAG
    DATA_DOMAIN = 'Client Reporting', CONFIDENTIALITY = 'Internal',
    DATA_STEWARD = 'DWS Data Engineering';

ALTER TABLE F_HOLDINGS_WITH_MARKET_DATA SET TAG
    DATA_DOMAIN = 'Client Reporting', CONFIDENTIALITY = 'Internal',
    DATA_STEWARD = 'DWS Data Engineering';

ALTER TABLE D_CLIENT SET TAG
    DATA_DOMAIN = 'Client Reporting', CONFIDENTIALITY = 'Confidential',
    DATA_STEWARD = 'DWS Client Onboarding';

SELECT * FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('F_POSITIONS_DAILY', 'TABLE'));

### 2b. Create a Share for Internal Publishing

A **Share** is how you package data for other Snowflake accounts (or other teams within your organisation). You grant access to specific tables, and consumers get a read-only view with zero data movement.

This is the same technology that powers the public Marketplace — except here we use it for internal data products within the organisation.

In [ ]:
CREATE OR REPLACE SHARE SNOWCAMP_CLIENT_REPORTING_SHARE
    COMMENT = 'Client Reporting Data Product - SnowCamp Lab';

GRANT USAGE ON DATABASE SNOWCAMP_LAB TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;
GRANT USAGE ON SCHEMA SNOWCAMP_LAB.MARTS TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;
GRANT SELECT ON TABLE SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY   TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;
GRANT SELECT ON TABLE SNOWCAMP_LAB.MARTS.F_HOLDINGS_WITH_MARKET_DATA TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;
GRANT SELECT ON TABLE SNOWCAMP_LAB.MARTS.D_PORTFOLIO         TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;
GRANT SELECT ON TABLE SNOWCAMP_LAB.MARTS.D_CLIENT            TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;

SHOW SHARES LIKE 'SNOWCAMP%';

### Publishing to the Internal Marketplace (UI)

Now let's publish your share as a discoverable **data product** in the Internal Marketplace. Open a **new browser tab** so you don't interrupt your notebook session.

#### Step 1: Navigate to Internal Sharing

In the Snowsight sidebar, go to **Horizon Catalog** → **Data sharing** → **Internal sharing**.

#### Step 2: Create a Listing

1. Click the **Listings** tab at the top.
2. Click **Create Listing**.
3. For the listing title, enter: `DWS Client Reporting Data Product - YOUR_FIRST_NAME YOUR_LAST_NAME`

#### Step 3: Add Your Data Product

1. Click the blue **Add Data Product** button.
2. Click **+ Select**, then find and select `SNOWCAMP_CLIENT_REPORTING_SHARE`.
3. Click **Save**.

#### Step 4: Configure Access Control

1. Click **+ Access Control**.
2. For **"Who can access this data product?"** — choose **No accounts or roles are pre-approved** (data will only be available by request).
3. For **"Who else can discover the listing and request access?"** — choose **Selected accounts and roles**.
4. From the dropdown, find and select **your own account**.

> **Note**: In a real deployment, this is where you would choose the specific accounts or roles that should be able to discover and request access to your data product.

#### Step 5: Set Up Request Approval

1. Click **Set up request approval flow**.
2. For **"How should the request approval happen?"** — choose **Manage Requests in Snowflake**.
3. For **"Approver email for notifications"** — select **Use Custom Email** and enter your own email address.
4. Click **Save** to return to the listing page.

#### Step 6: Add Support Contact and Publish

1. Click **Add Support Contact** and enter your own email address.
2. Click the **Publish** button in the top right corner.
3. Click **Done** in the confirmation prompt.

#### Step 7: Find Your Published Listing

1. In the sidebar, go to **Horizon Catalog** → **Catalog** → **Internal Marketplace**.
2. Sort listings by **Most Recent** (the sort button is next to the Create Listing button) and find your listing.

Congratulations — you have built and shared your own data product! Other teams in your organisation can now discover and request access to it, just like you installed the Snowflake Public Data listing in Day 1.

> **Reference**: [Creating Listings](https://docs.snowflake.com/en/user-guide/data-marketplace-listings) | [Internal Marketplace](https://docs.snowflake.com/en/user-guide/data-marketplace-internal)

---
## Part 3: Platform Administration & Monitoring (20 min)

As a platform team, you are not just building data products — you are also responsible for keeping the platform healthy, cost-effective, and secure. Snowflake provides rich observability through the `SNOWFLAKE.ACCOUNT_USAGE` schema, the Snowsight UI, and configurable resource monitors.

In this section we will explore the queries and tools that a DWS platform administrator would use daily.

> **Reference**: [ACCOUNT_USAGE Schema](https://docs.snowflake.com/en/sql-reference/account-usage) | [Resource Monitors](https://docs.snowflake.com/en/user-guide/resource-monitors) | [Query Insights](https://docs.snowflake.com/en/user-guide/ui-snowsight/activity)

### 3a. Credit Usage

Credits are Snowflake's unit of compute cost. Every query that runs on a warehouse consumes credits. Understanding where credits are being spent is the first step to cost optimisation — are there runaway queries? Is a warehouse auto-resuming too frequently? Should we downsize a warehouse that is overprovisioned?

In [ ]:
SELECT warehouse_name,
    SUM(credits_used) AS total_credits,
    SUM(credits_used_compute) AS compute_credits,
    SUM(credits_used_cloud_services) AS cloud_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE start_time >= DATEADD('hour', -2, CURRENT_TIMESTAMP())
GROUP BY warehouse_name ORDER BY total_credits DESC;

### 3b. Query History

The query history lets you identify slow queries, high-scan queries, and failed executions. This is your primary tool for performance troubleshooting. Look for queries with high `elapsed_sec` or `mb_scanned` — those are candidates for optimisation (better clustering, materialised views, or query rewriting).

In [ ]:
SELECT query_id, query_type, warehouse_name, execution_status,
    ROUND(total_elapsed_time / 1000, 2) AS elapsed_sec,
    ROUND(bytes_scanned / 1e6, 2) AS mb_scanned,
    rows_produced, query_text
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD('hour', -2, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
ORDER BY total_elapsed_time DESC LIMIT 15;

### 3c. Resource Monitors

Resource monitors are your safety net against runaway costs. They track credit consumption against a quota and can notify or even **suspend** a warehouse when the quota is reached. Every production environment should have these configured.

The monitor below sets a daily quota of 5 credits, notifies at 75% and 90%, and suspends the warehouse at 100%. In a real deployment, you would set these based on your budget and expected workload.

In [ ]:
CREATE OR REPLACE RESOURCE MONITOR LAB_MONITOR
    WITH CREDIT_QUOTA = 5 FREQUENCY = DAILY START_TIMESTAMP = IMMEDIATELY
    TRIGGERS
        ON 75 PERCENT DO NOTIFY
        ON 90 PERCENT DO NOTIFY
        ON 100 PERCENT DO SUSPEND;

ALTER WAREHOUSE WH_LAB SET RESOURCE_MONITOR = LAB_MONITOR;
SHOW RESOURCE MONITORS;

### Snowsight UI: Monitoring & Governance

Beyond SQL, Snowsight provides rich visual tools for platform management. Take a few minutes to explore these in the UI:

| Feature | Where to Find It | What It Shows |
|---------|-----------------|--------------|
| **Query History** | Monitoring → Query history | Slow queries, scan efficiency, execution plans |
| **Performance Explorer** | Monitoring → Performance explorer | Load patterns, queuing, auto-scaling decisions |
| **Cost Management** | Admin → Cost management | Credit usage trends, resource monitor status |
| **Trust Center** | Governance & security → Trust Center | Security posture, encryption status, network policies |
| **Tags & Policies** | Governance & security → Tags & policies | Tag-based masking policies, row access policies |

> **Reference**: [Query Insights](https://docs.snowflake.com/en/user-guide/ui-snowsight/activity) | [Cost Management](https://docs.snowflake.com/en/user-guide/cost-management) | [Trust Center](https://docs.snowflake.com/en/user-guide/ui-snowsight/trust-center)

In [ ]:
-- Review login activity — useful for security auditing
SELECT user_name, event_type, is_success, client_ip,
    first_authentication_factor, reported_client_type, event_timestamp
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY
WHERE event_timestamp >= DATEADD('hour', -3, CURRENT_TIMESTAMP())
ORDER BY event_timestamp DESC LIMIT 20;

---
## Part 4: Cortex Code — AI-Assisted Development in Snowflake (10 min)

Over the past two days you have been writing SQL and Python by hand. But Snowflake now includes an **AI coding assistant** called [Cortex Code](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code-snowsight) that can help you work faster and explore your data more effectively.

Cortex Code is not a chatbot that gives generic answers — it is an **agentic assistant** that understands your Snowflake environment: your roles, schemas, tables, and SQL syntax. It can generate queries, modify existing code, explain complex SQL, and even help with account administration.

### How to Access Cortex Code

1. Look for the **Cortex Code icon** in the lower-right corner of Snowsight
2. Click it to open the assistant panel
3. Type a question or request in natural language
4. Review the generated SQL — you can execute it directly or copy it to your worksheet

### Access Requirements

To use Cortex Code, your role needs these database roles granted:

```sql
GRANT DATABASE ROLE SNOWFLAKE.COPILOT_USER TO ROLE ACCOUNTADMIN;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER  TO ROLE ACCOUNTADMIN;
```

> **Reference**: [Cortex Code in Snowsight](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code-snowsight) | [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

### Ideas to Try with Cortex Code

Now that you have a fully built environment with databases, schemas, marts, a Streamlit app, and a dbt project, you have a rich playground for Cortex Code. Here are some prompts to try — open the Cortex Code panel and paste these in:

#### SQL Development & Exploration

| Try This Prompt | What It Does |
|----------------|-------------|
| *"Show me the top 10 clients by AUM from F_POSITIONS_DAILY"* | Generates a GROUP BY query with the right column names from your schema |
| *"Write a query for the 7-day moving average of portfolio returns"* | Demonstrates window functions tailored to your data |
| *"Why is this query slow? Optimize it."* | Analyses execution plans and suggests improvements |
| *"Explain the SQL in the F_HOLDINGS_WITH_MARKET_DATA table definition"* | Walks through complex JOIN logic step by step |

#### Data Discovery & Governance

| Try This Prompt | What It Does |
|----------------|-------------|
| *"What databases do I have access to?"* | Inventories your environment based on your role |
| *"Find all tables tagged with CONFIDENTIALITY = 'Confidential'"* | Searches Horizon Catalog metadata |
| *"Show the lineage from RAW.HOLDINGS to MARTS.F_POSITIONS_DAILY"* | Traces upstream/downstream dependencies |
| *"List every table in the MARTS schema and summarise key columns"* | Automated documentation generation |

#### dbt Projects

| Try This Prompt | What It Does |
|----------------|-------------|
| *"Create a staging model for a new RISK_FACTORS source table"* | Scaffolds a dbt model with clean column naming |
| *"Add not_null and unique tests to the holdings model"* | Generates schema.yml test definitions |
| *"Create an incremental model for daily position snapshots"* | Implements dbt incremental logic with merge behaviour |
| *"Generate documentation for the snowcamp_client_reporting dbt project"* | Auto-generates model and column descriptions |

#### Infrastructure & Cost

| Try This Prompt | What It Does |
|----------------|-------------|
| *"Which warehouses are using the most credits this week?"* | Queries ACCOUNT_USAGE for cost analysis |
| *"Show me failed queries in the last hour and suggest fixes"* | Combines query history with diagnostic suggestions |
| *"Create a resource monitor for the WH_LAB warehouse with a 10-credit daily limit"* | Generates the DDL with appropriate triggers |

#### Notebooks & Streamlit

| Try This Prompt | What It Does |
|----------------|-------------|
| *"Build a notebook that does EDA on the F_POSITIONS_DAILY table"* | Creates cells with summary stats, distributions, and charts |
| *"Add a bar chart of AUM by sector to the current notebook"* | Generates a matplotlib or Streamlit chart cell |
| *"Create a Streamlit app for portfolio risk analysis"* | Scaffolds a full Streamlit application |

### Tips for Getting the Best Results

1. **Be specific** — include database, schema, and table names in your prompts. Cortex Code works best when it knows exactly which objects you mean.
2. **Iterate** — ask follow-up questions to refine the output. "Now add a filter for region" or "Make it an incremental model" work well.
3. **Use quick actions** — highlight SQL text in a Workspace file and use the quick actions menu for Fix, Format, Explain, and Edit.
4. **Review before running** — Cortex Code shows a diff view for changes. Always review the generated SQL before executing.

> **Reference**: [Cortex Code in Snowsight](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code-snowsight) | [Cortex Code agent for dbt Projects](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code-snowsight#cortex-code-agent-for-dbt-projects-on-snowflake)

---
## Congratulations!

Over two days you have built an **end-to-end client-reporting data product** on Snowflake:

| Day | What You Built | Snowflake Feature |
|-----|---------------|-------------------|
| 1 | Three-schema database (RAW → ANALYTICS → MARTS) | CREATE DATABASE / SCHEMA |
| 1 | Six synthetic source tables with real NASDAQ tickers | Snowpark Python |
| 1 | Marketplace integration with real stock prices | Snowflake Marketplace (zero-copy) |
| 1 | Staging → Intermediate → Mart transformation pipeline | SQL / dbt |
| 1 | dbt Project with automated tests and deployment | CREATE / EXECUTE DBT PROJECT |
| 1 | Scheduled orchestration with Snowflake Tasks | CREATE TASK |
| 2 | Interactive Streamlit dashboard | Streamlit in Snowflake |
| 2 | Horizon Catalog tags and Internal Marketplace listing | Object Tagging / Shares |
| 2 | Credit monitoring, query analysis, resource monitors | ACCOUNT_USAGE / Resource Monitors |
| 2 | AI-assisted development with Cortex Code | Cortex Code |

### Resources

| Topic | Link |
|-------|------|
| Snowflake Workspaces | [docs.snowflake.com/...workspaces](https://docs.snowflake.com/en/user-guide/ui-snowsight/workspaces) |
| Snowflake Notebooks | [docs.snowflake.com/...notebooks](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks) |
| dbt Projects on Snowflake | [docs.snowflake.com/...dbt](https://docs.snowflake.com/en/user-guide/ui-snowsight/dbt) |
| CREATE DBT PROJECT | [docs.snowflake.com/...create-dbt-project](https://docs.snowflake.com/en/sql-reference/sql/create-dbt-project) |
| Streamlit in Snowflake | [docs.snowflake.com/...streamlit](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit) |
| Horizon Catalog | [docs.snowflake.com/...horizon](https://docs.snowflake.com/en/user-guide/governance-horizon) |
| Snowflake Public Data (Free) | [Snowflake Marketplace](https://app.snowflake.com/marketplace/listing/GZTSZAS2KCS) |
| ACCOUNT_USAGE | [docs.snowflake.com/...account-usage](https://docs.snowflake.com/en/sql-reference/account-usage) |
| Cortex Code | [docs.snowflake.com/...cortex-code](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code-snowsight) |

> *Thank you for attending DWS SnowCamp!*

---
## Teardown (Optional)

Run the cell below **only** when you want to remove all lab objects from your account. All statements are commented out to prevent accidental execution. Uncomment and run when you are ready to clean up.

In [ ]:
-- =====================================================
-- UNCOMMENT THE LINES BELOW TO TEAR DOWN THE LAB
-- =====================================================

-- Suspend and drop the scheduled Task
-- ALTER TASK SNOWCAMP_LAB.ANALYTICS.DBT_HOURLY_BUILD SUSPEND;
-- DROP TASK IF EXISTS SNOWCAMP_LAB.ANALYTICS.DBT_HOURLY_BUILD;

-- Drop the Streamlit app
-- DROP STREAMLIT IF EXISTS SNOWCAMP_LAB.ANALYTICS.CLIENT_REPORTING_APP;

-- Drop the dbt project
-- DROP DBT PROJECT IF EXISTS SNOWCAMP_LAB.ANALYTICS.SNOWCAMP_CLIENT_REPORTING;

-- Drop the share
-- DROP SHARE IF EXISTS SNOWCAMP_CLIENT_REPORTING_SHARE;

-- Drop the resource monitor
-- ALTER WAREHOUSE WH_LAB UNSET RESOURCE_MONITOR;
-- DROP RESOURCE MONITOR IF EXISTS LAB_MONITOR;

-- Drop the Git repository and API integration
-- DROP GIT REPOSITORY IF EXISTS SNOWCAMP_LAB.GITREPO.SNOWCAMP_GIT_REPO;
-- DROP API INTEGRATION IF EXISTS snowcamp_git_api;

-- Drop the lab database (removes ALL schemas, tables, views)
-- DROP DATABASE IF EXISTS SNOWCAMP_LAB;

-- Drop the warehouse
-- DROP WAREHOUSE IF EXISTS WH_LAB;

-- Drop the Marketplace database (if you no longer need it)
-- DROP DATABASE IF EXISTS FINANCIAL__ECONOMIC_ESSENTIALS;

SELECT 'Teardown complete — all lab objects removed.' AS status;